# WideResNet50-2 Teacher-Student Distillation (Layer2 Self-Contained)

This notebook is **self-contained**: it inlines the dataset loader, scoring helpers, model definition, training loop, evaluation, and optional score sweep.
You do **not** need extra helper modules or TOML files.

You still need the **processed dataset files** from your repo, especially the metadata CSV and `.npy` wafer arrays.

The notebook keeps the same comparison protocol as the rest of your repo:

- train on **normal wafers only**
- derive the deployed threshold from **validation normal** scores
- evaluate on the **shared test** split
- also save a **threshold sweep** for analysis only

In [ ]:

from pathlib import Path
import json
import math
import random
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from torchvision.models import Wide_ResNet50_2_Weights, wide_resnet50_2

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

In [ ]:
CONFIG = {
    "run": {
        "output_dir": "experiments/anomaly_detection/teacher_student/wideresnet50_2/x64/layer2_self_contained/artifacts/ts_wideresnet50_layer2",
        "seed": 42,
        "run_training": False,
        "run_score_sweep": False,
    },
    "data": {
        "repo_root": ".",
        "metadata_csv": "data/processed/x64/wm811k/metadata_50k_5pct.csv",
        "image_size": 64,
        "batch_size": 64,
        "num_workers": 8,
    },
    "training": {
        "epochs": 30,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5,
        "device": "cuda",
        "early_stopping_patience": 5,
        "early_stopping_min_delta": 1e-4,
        "checkpoint_every": 5,
    },
    "model": {
        "teacher_backbone": "wideresnet50_2",
        "teacher_layer": "layer2",
        "teacher_pretrained": True,
        "teacher_input_size": 224,
        "normalize_teacher_input": True,
        "feature_autoencoder_hidden_dim": 128,
        "student_weight": 1.0,
        "autoencoder_weight": 1.0,
        "score_student_weight": 1.0,
        "score_autoencoder_weight": 0.0,
        "reduction": "topk_mean",
        "topk_ratio": 0.20,
    },
    "scoring": {
        "threshold_quantile": 0.95,
    }
}

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

RUN_TRAINING = bool(CONFIG["run"].get("run_training", False))
RUN_SCORE_SWEEP = bool(CONFIG["run"].get("run_score_sweep", False))

image_size = int(CONFIG["data"]["image_size"])
batch_size = int(CONFIG["data"]["batch_size"])
num_workers = int(CONFIG["data"]["num_workers"])
requested_device_name = str(CONFIG["training"]["device"])
device = torch.device("cuda" if requested_device_name == "auto" and torch.cuda.is_available() else requested_device_name)

seed = int(CONFIG["run"]["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

CONFIG

In [ ]:

# -----------------------------
# Shared helpers
# -----------------------------
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)

def spatial_mean(error_map: torch.Tensor) -> torch.Tensor:
    dims = tuple(range(1, error_map.ndim))
    return error_map.mean(dim=dims)

def spatial_max(error_map: torch.Tensor) -> torch.Tensor:
    flattened = error_map.flatten(start_dim=1)
    return flattened.max(dim=1).values

def topk_spatial_mean(error_map: torch.Tensor, topk_ratio: float) -> torch.Tensor:
    if not 0.0 < topk_ratio <= 1.0:
        raise ValueError(f"topk_ratio must be in (0, 1], got {topk_ratio}")
    flattened = error_map.flatten(start_dim=1)
    topk = max(1, int(math.ceil(flattened.shape[1] * topk_ratio)))
    return torch.topk(flattened, k=topk, dim=1).values.mean(dim=1)

def summarize_threshold_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    predicted = (scores > threshold).astype(int)
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "auroc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "predicted_anomalies": int(predicted.sum()),
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
    }

def sweep_threshold_metrics(labels: np.ndarray, scores: np.ndarray):
    precision_curve, recall_curve, thresholds = precision_recall_curve(labels, scores)
    if thresholds.size == 0:
        sweep_df = pd.DataFrame([{
            "threshold": float(scores[0]) if scores.size else 0.0,
            "precision": float(precision_curve[0]),
            "recall": float(recall_curve[0]),
            "f1": 0.0,
            "predicted_anomalies": int((scores > 0).sum()),
        }])
    else:
        sweep_df = pd.DataFrame({
            "threshold": thresholds,
            "precision": precision_curve[:-1],
            "recall": recall_curve[:-1],
        })
        sweep_df["f1"] = 2.0 * sweep_df["precision"] * sweep_df["recall"] / (
            sweep_df["precision"] + sweep_df["recall"] + 1e-12
        )
        sweep_df["predicted_anomalies"] = [(scores > t).sum() for t in sweep_df["threshold"]]
    best_row = sweep_df.loc[sweep_df["f1"].idxmax()].to_dict()
    return sweep_df, {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in best_row.items()}

set_seed(int(CONFIG["run"]["seed"]))
device = resolve_device(str(CONFIG["training"].get("device", "auto")))
print("Resolved device:", device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# =========================
# HELPER FUNCTIONS + PREPARE DATASET FROM LSWMD.pkl
# same protocol as the local training notebook
# =========================

import os
import sys
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pandas.core.indexes as core_indexes
import torch
import torch.nn.functional as F

LABEL_NORMAL = "none"
LABEL_DEFECT = "pattern"

RAW_PICKLE = str(REPO_ROOT / "data" / "raw" / "LSWMD.pkl")   # change if needed
OUTPUT_ROOT = str(REPO_ROOT)
IMAGE_SIZE = 64
NORMAL_LIMIT = 50000
TEST_DEFECT_FRACTION = 0.05
SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def read_legacy_pickle(path: Path) -> pd.DataFrame:
    sys.modules["pandas.indexes"] = core_indexes
    with path.open("rb") as handle:
        return pickle.load(handle, encoding="latin1")

def unwrap_legacy_value(value):
    if value is None:
        return ""
    if hasattr(value, "size") and getattr(value, "size") == 0:
        return ""
    if hasattr(value, "tolist"):
        value = value.tolist()
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    return str(value).strip()

def normalize_map(wafer_map: np.ndarray, image_size: int) -> np.ndarray:
    wafer_map = np.asarray(wafer_map, dtype=np.float32)
    wafer_map = wafer_map / 2.0
    tensor = torch.from_numpy(wafer_map).unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(tensor, size=(image_size, image_size), mode="nearest")
    return resized.squeeze(0).squeeze(0).numpy()

def infer_label_from_row(row: pd.Series):
    failure = unwrap_legacy_value(row.get("failureType", "")).lower()
    if failure == "none":
        return LABEL_NORMAL
    if failure:
        return LABEL_DEFECT
    return None

def split_normals(normal_df: pd.DataFrame, seed: int) -> pd.DataFrame:
    shuffled = normal_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(shuffled)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)
    shuffled.loc[: train_end - 1, "split"] = "train"
    shuffled.loc[train_end: val_end - 1, "split"] = "val"
    shuffled.loc[val_end:, "split"] = "test"
    return shuffled

def sample_test_defects(defect_df: pd.DataFrame, normal_df: pd.DataFrame, fraction: float, seed: int) -> pd.DataFrame:
    test_normal_count = int((normal_df["split"] == "test").sum())
    requested = max(1, int(round(test_normal_count * fraction)))
    sampled = defect_df.sample(n=min(requested, len(defect_df)), random_state=seed).copy()
    sampled["split"] = "test"
    return sampled

set_seed(SEED)

output_root = Path(OUTPUT_ROOT)
processed_dir = output_root / "data" / "processed" / "x64" / "wm811k"
arrays_dir = processed_dir / "arrays"
metadata_path = processed_dir / "metadata_50k_5pct.csv"

processed_dir.mkdir(parents=True, exist_ok=True)
arrays_dir.mkdir(parents=True, exist_ok=True)

print(f"Using raw pickle: {RAW_PICKLE}")
print(f"Saving arrays to: {arrays_dir}")
print(f"Saving metadata to: {metadata_path}")

df = read_legacy_pickle(Path(RAW_PICKLE))
df = df.copy()
df["failureTypeText"] = df["failureType"].map(unwrap_legacy_value)
df["trianTestLabelText"] = df["trianTestLabel"].map(unwrap_legacy_value)
df["label"] = df.apply(infer_label_from_row, axis=1)
df = df[df["label"].notna()].reset_index(drop=True)

normal_df = df[df["label"] == LABEL_NORMAL].copy()
defect_df = df[df["label"] == LABEL_DEFECT].copy()

normal_df = normal_df.sample(n=min(NORMAL_LIMIT, len(normal_df)), random_state=SEED)
normal_df = split_normals(normal_df, SEED)
defect_df = sample_test_defects(defect_df, normal_df, TEST_DEFECT_FRACTION, SEED)

export_df = pd.concat([normal_df, defect_df], ignore_index=True)

print(
    f"Preparing arrays | normals={len(normal_df)} | defects={len(defect_df)} | total={len(export_df)}",
    flush=True,
)

records = []

for row_index, row in export_df.iterrows():
    array_path = arrays_dir / f"wafer_{row_index:06d}.npy"
    wafer_array = normalize_map(np.asarray(row["waferMap"]), IMAGE_SIZE)
    np.save(array_path, wafer_array)
    records.append(
        {
            "array_path": str(array_path.resolve()),
            "failure_type": row["failureTypeText"],
            "failureTypeText": row["failureTypeText"],
            "defect_type": row["failureTypeText"],
            "label": row["label"],
            "is_anomaly": int(row["label"] == LABEL_DEFECT),
            "split": row["split"],
        }
    )

metadata_df = pd.DataFrame(records)
metadata_df.to_csv(metadata_path, index=False)

print("Saved processed metadata:", metadata_path)
display(metadata_df.head())
display(metadata_df.groupby(["split", "is_anomaly"]).size().rename("count").reset_index())

In [ ]:

# -----------------------------
# Dataset loader (inlined)
# -----------------------------
class WaferMapDataset(Dataset):
    def __init__(self, metadata_csv: str | Path, split: str, image_size: int = 64) -> None:
        self.metadata_path = self._resolve_metadata_path(Path(metadata_csv), image_size).resolve()
        self.repo_root = self._find_repo_root(self.metadata_path)
        self.metadata = pd.read_csv(self.metadata_path)
        self.metadata = self.metadata[self.metadata["split"] == split].reset_index(drop=True)
        self.image_size = image_size

    @staticmethod
    def _candidate_repo_roots() -> list[Path]:
        roots = []
        cwd = Path.cwd().resolve()
        roots.extend([cwd, *cwd.parents])
        return list(dict.fromkeys(roots))

    @staticmethod
    def _resolve_metadata_path(metadata_path: Path, image_size: int) -> Path:
        if metadata_path.exists():
            return metadata_path
        cwd_candidate = (Path.cwd() / metadata_path).resolve()
        if cwd_candidate.exists():
            return cwd_candidate
        matches = []
        for repo_root in WaferMapDataset._candidate_repo_roots():
            direct_candidate = (repo_root / metadata_path).resolve()
            if direct_candidate.exists():
                return direct_candidate
            processed_root = repo_root / "data" / "processed"
            if processed_root.exists():
                matches.extend(processed_root.rglob(metadata_path.name))
        unique_matches = list(dict.fromkeys(matches))
        if len(unique_matches) == 1:
            return unique_matches[0]
        if len(unique_matches) > 1:
            size_matches = [m for m in unique_matches if f"x{image_size}" in m.parts]
            if len(size_matches) == 1:
                return size_matches[0]
        raise FileNotFoundError(f"Could not resolve metadata CSV: {metadata_path}")

    @staticmethod
    def _find_repo_root(metadata_path: Path) -> Path:
        for parent in [metadata_path.parent, *metadata_path.parents]:
            if (parent / "data").exists():
                return parent
        return metadata_path.parent

    def __len__(self) -> int:
        return len(self.metadata)

    def __getitem__(self, index: int):
        row = self.metadata.iloc[index]
        array_rel = row["array_path"]
        array_path = (self.repo_root / array_rel).resolve()
        if not array_path.exists():
            raise FileNotFoundError(f"Missing wafer array file: {array_path}")
        wafer = np.load(array_path).astype(np.float32)
        if wafer.ndim == 2:
            wafer = wafer[None, :, :]
        label = int(row["is_anomaly"])
        return torch.from_numpy(wafer), torch.tensor(label, dtype=torch.long)

image_size = int(CONFIG["data"]["image_size"])
batch_size = int(CONFIG["data"]["batch_size"])
num_workers = int(CONFIG["data"].get("num_workers", 0))
metadata_path = REPO_ROOT / CONFIG["data"]["metadata_csv"]

train_dataset = WaferMapDataset(metadata_path, split="train", image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split="val", image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split="test", image_size=image_size)

pin_memory = device.type == "cuda"
persistent_workers = num_workers > 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                          pin_memory=pin_memory, persistent_workers=persistent_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                        pin_memory=pin_memory, persistent_workers=persistent_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                         pin_memory=pin_memory, persistent_workers=persistent_workers)

split_summary = pd.DataFrame([
    {"split": "train", "count": len(train_dataset), "anomalies": int(train_dataset.metadata["is_anomaly"].sum())},
    {"split": "val", "count": len(val_dataset), "anomalies": int(val_dataset.metadata["is_anomaly"].sum())},
    {"split": "test", "count": len(test_dataset), "anomalies": int(test_dataset.metadata["is_anomaly"].sum())},
])
display(split_summary)
display(train_dataset.metadata.head())

In [ ]:
# -----------------------------
# WideResNet50-2 teacher + TS model
# -----------------------------
def _teacher_feature_dim(backbone_name: str, layer_name: str) -> int:
    dims = {
        "wideresnet50_2": {"layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},
        "wide_resnet50_2": {"layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},
    }
    if backbone_name not in dims or layer_name not in dims[backbone_name]:
        raise ValueError(f"Unsupported teacher backbone/layer combination: {backbone_name} / {layer_name}")
    return dims[backbone_name][layer_name]

class WideResNet50_2Teacher(nn.Module):
    def __init__(
        self,
        pretrained: bool = True,
        input_size: int = 224,
        freeze_backbone: bool = True,
        normalize_imagenet: bool = True,
    ):
        super().__init__()
        weights = Wide_ResNet50_2_Weights.DEFAULT if pretrained else None
        backbone = wide_resnet50_2(weights=weights)

        original_conv = backbone.conv1
        adapted_conv = nn.Conv2d(
            1,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            adapted_conv.weight.copy_(original_conv.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = adapted_conv

        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.input_size = int(input_size)
        self.normalize_imagenet = bool(normalize_imagenet)

        self.register_buffer("image_mean", torch.tensor([0.4490], dtype=torch.float32).view(1, 1, 1, 1))
        self.register_buffer("image_std", torch.tensor([0.2260], dtype=torch.float32).view(1, 1, 1, 1))

        if freeze_backbone:
            for p in self.parameters():
                p.requires_grad = False

    def preprocess(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.input_size or x.shape[-2] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        if self.normalize_imagenet:
            x = (x - self.image_mean) / self.image_std
        return x

    def forward_intermediate_feature_map(self, x: torch.Tensor, layer_name: str = "layer2") -> torch.Tensor:
        x = self.preprocess(x)
        x = self.stem(x)
        x = self.layer1(x)
        if layer_name == "layer1":
            return x
        x = self.layer2(x)
        if layer_name == "layer2":
            return x
        x = self.layer3(x)
        if layer_name == "layer3":
            return x
        x = self.layer4(x)
        return x

class TSStudent(nn.Module):
    def __init__(self, feature_dim: int, input_size: int = 224) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, feature_dim, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),

            nn.Conv2d(feature_dim, feature_dim, kernel_size=3, stride=1, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

class TeacherFeatureAutoencoder(nn.Module):
    def __init__(self, feature_dim: int, hidden_dim: int = 128) -> None:
        super().__init__()
        bottleneck_dim = max(hidden_dim // 2, 16)
        self.encoder = nn.Sequential(
            nn.Conv2d(feature_dim, hidden_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, bottleneck_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(bottleneck_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden_dim, feature_dim, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

class WideTeacherTSDistillationModel(nn.Module):
    def __init__(self, config: dict, image_size: int = 64):
        super().__init__()
        model_config = config["model"]
        self.teacher_backbone = str(model_config.get("teacher_backbone", "wideresnet50_2")).lower()
        self.teacher_layer = str(model_config.get("teacher_layer", "layer2")).lower()
        self.teacher_input_size = int(model_config.get("teacher_input_size", 224))
        self.reduction = str(model_config.get("reduction", "topk_mean"))
        self.topk_ratio = float(model_config.get("topk_ratio", 0.20))
        self.student_weight = float(model_config.get("student_weight", 1.0))
        self.autoencoder_weight = float(model_config.get("autoencoder_weight", 1.0))
        self.score_student_weight = float(model_config.get("score_student_weight", self.student_weight))
        self.score_autoencoder_weight = float(model_config.get("score_autoencoder_weight", self.autoencoder_weight))
        self.feature_dim = _teacher_feature_dim(self.teacher_backbone, self.teacher_layer)

        self.teacher = WideResNet50_2Teacher(
            pretrained=bool(model_config.get("teacher_pretrained", True)),
            input_size=self.teacher_input_size,
            freeze_backbone=True,
            normalize_imagenet=bool(model_config.get("normalize_teacher_input", True)),
        )
        self.student = TSStudent(feature_dim=self.feature_dim, input_size=self.teacher_input_size)
        self.autoencoder = TeacherFeatureAutoencoder(
            feature_dim=self.feature_dim,
            hidden_dim=int(model_config.get("feature_autoencoder_hidden_dim", 128)),
        )

        self.register_buffer("student_map_scale", torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer("autoencoder_map_scale", torch.tensor(1.0, dtype=torch.float32))

    def teacher_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            features = self.teacher.forward_intermediate_feature_map(x, layer_name=self.teacher_layer)

        # Critical stabilization:
        # Normalize frozen teacher features before comparing with student / AE outputs.
        # This prevents huge squared-error values from exploding the AE loss.
        features = F.layer_norm(features, features.shape[1:])
        return features

    def _student_input(self, x: torch.Tensor) -> torch.Tensor:
        return self.teacher.preprocess(x)

    def student_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        return self.student(self._student_input(x))

    def autoencoder_feature_map(self, teacher_features: torch.Tensor) -> torch.Tensor:
        autoencoder_features = self.autoencoder(teacher_features)
        if autoencoder_features.shape[-2:] != teacher_features.shape[-2:]:
            autoencoder_features = F.interpolate(
                autoencoder_features,
                size=teacher_features.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return autoencoder_features

    def raw_anomaly_maps(self, x: torch.Tensor):
        teacher_features = self.teacher_feature_map(x)
        student_features = self.student_feature_map(x)

        if student_features.shape[-2:] != teacher_features.shape[-2:]:
            student_features = F.interpolate(
                student_features,
                size=teacher_features.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        autoencoder_features = self.autoencoder_feature_map(teacher_features)

        # Use mean squared channel error maps on normalized teacher features.
        # Clamp only the maps, not the features, to avoid runaway loss values.
        student_map = torch.mean((student_features - teacher_features).pow(2), dim=1, keepdim=True)
        autoencoder_map = torch.mean((autoencoder_features - teacher_features).pow(2), dim=1, keepdim=True)

        student_map = torch.clamp(student_map, min=0.0, max=100.0)
        autoencoder_map = torch.clamp(autoencoder_map, min=0.0, max=100.0)

        return student_map, autoencoder_map

    def set_error_scales(self, student_scale: float, autoencoder_scale: float) -> None:
        self.student_map_scale.fill_(max(float(student_scale), 1e-6))
        self.autoencoder_map_scale.fill_(max(float(autoencoder_scale), 1e-6))

    def normalized_anomaly_map(self, x: torch.Tensor) -> torch.Tensor:
        student_map, autoencoder_map = self.raw_anomaly_maps(x)
        return (
            self.score_student_weight * (student_map / self.student_map_scale.clamp_min(1e-6))
            + self.score_autoencoder_weight * (autoencoder_map / self.autoencoder_map_scale.clamp_min(1e-6))
        )

    def reduce_anomaly_map(self, anomaly_map: torch.Tensor) -> torch.Tensor:
        if self.reduction == "mean":
            return spatial_mean(anomaly_map)
        if self.reduction == "max":
            return spatial_max(anomaly_map)
        return topk_spatial_mean(anomaly_map, self.topk_ratio)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.reduce_anomaly_map(self.normalized_anomaly_map(x))

In [ ]:
# -----------------------------
# Training helpers
# -----------------------------
def clone_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

class EpochMetrics:
    def __init__(self, loss: float, distillation_loss: float, auxiliary_loss: float) -> None:
        self.loss = float(loss)
        self.distillation_loss = float(distillation_loss)
        self.auxiliary_loss = float(auxiliary_loss)

def run_ts_epoch(model: nn.Module, dataloader: DataLoader, device: torch.device, optimizer=None, desc: str | None = None):
    is_training = optimizer is not None
    model.train(is_training)
    model.teacher.eval()

    total_loss = 0.0
    total_student_loss = 0.0
    total_auto_loss = 0.0
    total_items = 0

    iterator = tqdm(dataloader, desc=desc) if desc else dataloader

    for inputs, labels in iterator:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        normal_mask = labels == 0
        if not torch.any(normal_mask):
            continue

        normal_inputs = inputs[normal_mask]

        # autocast only for forward pass
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            student_map, auto_map = model.raw_anomaly_maps(normal_inputs)

            student_loss = student_map.mean()
            auto_loss = auto_map.mean()

            loss = model.student_weight * student_loss + model.autoencoder_weight * auto_loss

        if is_training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        batch_size = normal_inputs.shape[0]

        total_loss += float(loss.detach().item()) * batch_size
        total_student_loss += float(student_loss.detach().item()) * batch_size
        total_auto_loss += float(auto_loss.detach().item()) * batch_size
        total_items += batch_size

    denom = max(total_items, 1)

    return EpochMetrics(
        loss=total_loss / denom,
        distillation_loss=total_student_loss / denom,
        auxiliary_loss=total_auto_loss / denom,
    )

def estimate_ts_error_scales(model: nn.Module, dataloader: DataLoader, device: torch.device):
    model.eval()

    total_student = 0.0
    total_auto = 0.0
    total_items = 0

    with torch.inference_mode():
        iterator = tqdm(dataloader, desc="Estimate error scales")

        for inputs, labels in iterator:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            normal_mask = labels == 0
            if not torch.any(normal_mask):
                continue

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                student_map, auto_map = model.raw_anomaly_maps(inputs[normal_mask])

            batch_size = int(normal_mask.sum().item())

            total_student += float(student_map.mean().item()) * batch_size
            total_auto += float(auto_map.mean().item()) * batch_size
            total_items += batch_size

    if total_items == 0:
        raise ValueError("Could not estimate TS error scales because no normal samples were found.")

    return max(total_student / total_items, 1e-6), max(total_auto / total_items, 1e-6)

In [ ]:
# -----------------------------
# Build model and optimizer
# -----------------------------
output_dir = REPO_ROOT / CONFIG["run"]["output_dir"]
checkpoint_dir = output_dir / "checkpoints"
results_dir = output_dir / "results"
evaluation_dir = results_dir / "evaluation"
plots_dir = output_dir / "plots"

for path in [output_dir, checkpoint_dir, results_dir, evaluation_dir, plots_dir]:
    path.mkdir(parents=True, exist_ok=True)

# Save config for reproducibility
with (results_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print("Experiment outputs saved to:", output_dir)

model = WideTeacherTSDistillationModel(CONFIG, image_size=image_size).to(device)
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=float(CONFIG["training"]["learning_rate"]),
    weight_decay=float(CONFIG["training"]["weight_decay"]),
)

print("Model device:", next(model.parameters()).device)
print("Teacher layer:", CONFIG["model"]["teacher_layer"])
print("Feature dim:", model.feature_dim)

In [ ]:
# -----------------------------
# Train
# -----------------------------
history = []
best_val_loss = float("inf")
best_epoch = 0
best_state_dict = None

epochs = int(CONFIG["training"]["epochs"])
patience = int(CONFIG["training"].get("early_stopping_patience", 0))
min_delta = float(CONFIG["training"].get("early_stopping_min_delta", 0.0))
checkpoint_every = int(CONFIG["training"].get("checkpoint_every", 5))
stale_epochs = 0

checkpoint_dir = output_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

if RUN_TRAINING:
    for epoch in range(epochs):
        epoch_start = time.time()

        train_metrics = run_ts_epoch(
            model,
            train_loader,
            device,
            optimizer=optimizer,
            desc=f"Train {epoch+1}/{epochs}",
        )
        val_metrics = run_ts_epoch(
            model,
            val_loader,
            device,
            optimizer=None,
            desc=f"Val {epoch+1}/{epochs}",
        )

        record = {
            "epoch": epoch + 1,
            "train_loss": train_metrics.loss,
            "val_loss": val_metrics.loss,
            "train_distillation_loss": train_metrics.distillation_loss,
            "val_distillation_loss": val_metrics.distillation_loss,
            "train_autoencoder_loss": train_metrics.auxiliary_loss,
            "val_autoencoder_loss": val_metrics.auxiliary_loss,
            "epoch_seconds": time.time() - epoch_start,
        }
        history.append(record)
        print(record)

        improved = val_metrics.loss < (best_val_loss - min_delta)
        if improved:
            best_val_loss = val_metrics.loss
            best_epoch = epoch + 1
            best_state_dict = clone_state_dict(model)
            stale_epochs = 0

            torch.save(
                {
                    "config": CONFIG,
                    "epoch": best_epoch,
                    "model_state_dict": best_state_dict,
                    "best_val_loss": best_val_loss,
                },
                checkpoint_dir / "best_model.pt",
            )
            print(f"Saved best model: {checkpoint_dir / 'best_model.pt'}")
        else:
            stale_epochs += 1

        if checkpoint_every > 0 and ((epoch + 1) % checkpoint_every == 0):
            ckpt_path = checkpoint_dir / f"checkpoint_epoch_{epoch+1}.pt"
            torch.save(
                {
                    "config": CONFIG,
                    "epoch": epoch + 1,
                    "model_state_dict": clone_state_dict(model),
                    "val_loss": val_metrics.loss,
                },
                ckpt_path,
            )
            print(f"Saved checkpoint: {ckpt_path}")

        if patience > 0 and stale_epochs >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
else:
    print("Skipping training because RUN_TRAINING=False")

# Save final model state as well
final_epoch = history[-1]["epoch"] if history else 0
torch.save(
    {
        "config": CONFIG,
        "epoch": final_epoch,
        "model_state_dict": clone_state_dict(model),
    },
    output_dir / "final_model.pt",
)
print(f"Saved final model: {output_dir / 'final_model.pt'}")

history_path = output_dir / "history.json"
with history_path.open("w", encoding="utf-8") as handle:
    json.dump(history, handle, indent=2)

history_df = pd.DataFrame(history)
history_df.to_csv(output_dir / "training_history.csv", index=False)

training_summary = {
    "best_epoch": int(best_epoch) if best_epoch else None,
    "best_val_loss": float(best_val_loss) if math.isfinite(best_val_loss) else None,
    "history_path": str(history_path),
    "training_history_csv": str(output_dir / "training_history.csv"),
    "best_model_path": str(checkpoint_dir / "best_model.pt"),
    "final_model_path": str(output_dir / "final_model.pt"),
    "checkpoint_dir": str(checkpoint_dir),
    "epochs_completed": int(len(history)),
}
with (output_dir / "summary.json").open("w", encoding="utf-8") as handle:
    json.dump(training_summary, handle, indent=2)

display(history_df.tail())

In [ ]:
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val")
    axes[0].set_title("Wide TS total loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_distillation_loss"], marker="o", label="train student")
    axes[1].plot(history_df["epoch"], history_df["val_distillation_loss"], marker="o", label="val student")
    axes[1].plot(history_df["epoch"], history_df["train_autoencoder_loss"], marker="o", label="train auto")
    axes[1].plot(history_df["epoch"], history_df["val_autoencoder_loss"], marker="o", label="val auto")
    axes[1].set_title("Component losses")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(plots_dir / "training_curves.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("No training history to plot.")

In [ ]:
# -----------------------------
# Evaluate default configured score
# -----------------------------
summary_path = evaluation_dir / "summary.json"
val_scores_path = evaluation_dir / "val_scores.csv"
test_scores_path = evaluation_dir / "test_scores.csv"
threshold_sweep_path = evaluation_dir / "threshold_sweep.csv"
checkpoint_path = checkpoint_dir / "best_model.pt"

def collect_scores(model: nn.Module, dataloader: DataLoader, device: torch.device, desc: str):
    scores = []
    labels = []
    with torch.inference_mode():
        iterator = tqdm(dataloader, desc=desc)
        autocast_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if device.type == "cuda"
            else torch.autocast(device_type="cpu", enabled=False)
        )
        with autocast_context:
            for inputs, batch_labels in iterator:
                inputs = inputs.to(device, non_blocking=True)
                batch_scores = model(inputs)
                scores.append(batch_scores.detach().cpu())
                labels.append(batch_labels.cpu())
    return torch.cat(scores, dim=0).numpy(), torch.cat(labels, dim=0).numpy()

use_cached_evaluation = (
    (not RUN_TRAINING)
    and summary_path.exists()
    and val_scores_path.exists()
    and test_scores_path.exists()
    and threshold_sweep_path.exists()
)

if use_cached_evaluation:
    print(f"Loading cached evaluation from {evaluation_dir}")
    default_eval_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    val_scores_df = pd.read_csv(val_scores_path)
    test_scores_df = pd.read_csv(test_scores_path)
    threshold_sweep_df = pd.read_csv(threshold_sweep_path)
else:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    student_scale, auto_scale = estimate_ts_error_scales(model, val_loader, device)
    model.set_error_scales(student_scale, auto_scale)

    val_scores, val_labels = collect_scores(model, val_loader, device, desc="Val scores")
    test_scores, test_labels = collect_scores(model, test_loader, device, desc="Test scores")

    val_scores_df = pd.DataFrame({"score": val_scores, "is_anomaly": val_labels.astype(int)})
    test_scores_df = pd.DataFrame({"score": test_scores, "is_anomaly": test_labels.astype(int)})

    threshold_quantile = float(CONFIG["scoring"].get("threshold_quantile", 0.95))
    val_normal_scores = val_scores_df.loc[val_scores_df["is_anomaly"] == 0, "score"]
    threshold = float(val_normal_scores.quantile(threshold_quantile))

    metrics = summarize_threshold_metrics(test_labels.astype(int), test_scores, threshold)
    threshold_sweep_df, best_sweep = sweep_threshold_metrics(test_labels.astype(int), test_scores)

    val_scores_df.to_csv(val_scores_path, index=False)
    test_scores_df.to_csv(test_scores_path, index=False)
    threshold_sweep_df.to_csv(threshold_sweep_path, index=False)

    default_eval_summary = {
        "teacher_backbone": CONFIG["model"]["teacher_backbone"],
        "teacher_layer": CONFIG["model"]["teacher_layer"],
        "threshold_quantile": threshold_quantile,
        "threshold": threshold,
        "metrics_at_validation_threshold": metrics,
        "best_threshold_sweep": best_sweep,
    }
    summary_path.write_text(json.dumps(default_eval_summary, indent=2), encoding="utf-8")

default_eval_summary

In [ ]:

metrics_df = pd.DataFrame([
    {"metric": "precision", "value": default_eval_summary["metrics_at_validation_threshold"]["precision"]},
    {"metric": "recall", "value": default_eval_summary["metrics_at_validation_threshold"]["recall"]},
    {"metric": "f1", "value": default_eval_summary["metrics_at_validation_threshold"]["f1"]},
    {"metric": "auroc", "value": default_eval_summary["metrics_at_validation_threshold"]["auroc"]},
    {"metric": "auprc", "value": default_eval_summary["metrics_at_validation_threshold"]["auprc"]},
    {"metric": "threshold", "value": default_eval_summary["threshold"]},
])
display(metrics_df)

cm = np.array(default_eval_summary["metrics_at_validation_threshold"]["confusion_matrix"])
cm_df = pd.DataFrame(cm, index=["true_normal", "true_anomaly"], columns=["pred_normal", "pred_anomaly"])
display(cm_df)

best = default_eval_summary["best_threshold_sweep"]
print(
    f"Best sweep threshold: {best['threshold']:.6f} | "
    f"precision={best['precision']:.4f}, recall={best['recall']:.4f}, f1={best['f1']:.4f}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(val_scores_df["score"], bins=30, alpha=0.8, color="#4d908e")
axes[0].axvline(default_eval_summary["threshold"], color="red", linestyle="--", label=f"threshold={default_eval_summary['threshold']:.4f}")
axes[0].set_title("Validation score distribution")
axes[0].set_xlabel("Anomaly score")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(test_scores_df.loc[test_scores_df["is_anomaly"] == 0, "score"], bins=30, alpha=0.7, label="normal")
axes[1].hist(test_scores_df.loc[test_scores_df["is_anomaly"] == 1, "score"], bins=30, alpha=0.7, label="anomaly")
axes[1].axvline(default_eval_summary["threshold"], color="red", linestyle="--", label=f"threshold={default_eval_summary['threshold']:.4f}")
axes[1].set_title("Test score distribution")
axes[1].set_xlabel("Anomaly score")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
fig.savefig(plots_dir / "score_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
threshold_sweep_df.plot(x="threshold", y=["precision", "recall", "f1"], ax=ax)
ax.axvline(default_eval_summary["threshold"], color="red", linestyle="--", linewidth=1.75, label="validation threshold")
ax.set_title("WideResNet50-2 TS threshold sweep")
ax.set_ylabel("metric")
ax.legend()
plt.tight_layout()
fig.savefig(plots_dir / "threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["pred_normal", "pred_anomaly"])
ax.set_yticks([0, 1], labels=["true_normal", "true_anomaly"])
ax.set_title("Confusion matrix")
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, int(cm[row, col]), ha="center", va="center", color="#111111")
plt.tight_layout()
fig.savefig(plots_dir / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## Optional Score Sweep

This keeps the trained checkpoint fixed and sweeps different branch weights and wafer-level reductions under the **same validation-threshold evaluation rule**.

In [ ]:

SCORE_SWEEP_WEIGHTS = [
    (1.0, 0.0),
    (1.0, 1.0),
    (2.0, 1.0),
    (1.0, 2.0),
]
SCORE_SWEEP_REDUCTIONS = [
    ("topk_mean", 0.10),
    ("topk_mean", 0.15),
    ("topk_mean", 0.20),
    ("topk_mean", 0.25),
    ("mean", None),
    ("max", None),
]

def collect_normalized_maps(model: nn.Module, dataloader: DataLoader, device: torch.device, desc: str):
    student_maps, auto_maps, labels = [], [], []
    with torch.inference_mode():
        iterator = tqdm(dataloader, desc=desc)
        autocast_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if device.type == "cuda"
            else torch.autocast(device_type="cpu", enabled=False)
        )
        with autocast_context:
            for inputs, batch_labels in iterator:
                inputs = inputs.to(device, non_blocking=True)
                student_map, auto_map = model.raw_anomaly_maps(inputs)
                student_maps.append((student_map / model.student_map_scale.clamp_min(1e-6)).detach().cpu())
                auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-6)).detach().cpu())
                labels.append(batch_labels.cpu())
    return (
        torch.cat(student_maps, dim=0).numpy(),
        torch.cat(auto_maps, dim=0).numpy(),
        torch.cat(labels, dim=0).numpy(),
    )

def reduce_map_np(anomaly_map: np.ndarray, reduction: str, topk_ratio: float | None = None) -> np.ndarray:
    flat = anomaly_map.reshape(anomaly_map.shape[0], -1)
    if reduction == "mean":
        return flat.mean(axis=1).astype(np.float32)
    if reduction == "max":
        return flat.max(axis=1).astype(np.float32)
    if reduction == "topk_mean":
        if topk_ratio is None:
            raise ValueError("topk_ratio is required for topk_mean")
        k = max(1, int(math.ceil(flat.shape[1] * topk_ratio)))
        topk = np.partition(flat, kth=flat.shape[1]-k, axis=1)[:, -k:]
        return topk.mean(axis=1).astype(np.float32)
    raise ValueError(f"Unsupported reduction: {reduction}")

score_sweep_rows = []

if RUN_SCORE_SWEEP:
    model.eval()
    val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(model, val_loader, device, "Val normalized maps")
    test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(model, test_loader, device, "Test normalized maps")

    for student_weight, auto_weight in SCORE_SWEEP_WEIGHTS:
        for reduction, ratio in SCORE_SWEEP_REDUCTIONS:
            val_map = student_weight * val_student_maps + auto_weight * val_auto_maps
            test_map = student_weight * test_student_maps + auto_weight * test_auto_maps

            val_scores_variant = reduce_map_np(val_map, reduction, ratio)
            test_scores_variant = reduce_map_np(test_map, reduction, ratio)

            val_normal = val_scores_variant[val_labels == 0]
            if val_normal.size == 0:
                continue
            threshold_variant = float(pd.Series(val_normal).quantile(threshold_quantile))
            metrics_variant = summarize_threshold_metrics(test_labels.astype(int), test_scores_variant, threshold_variant)
            sweep_df_variant, best_variant = sweep_threshold_metrics(test_labels.astype(int), test_scores_variant)

            name = f"s{student_weight:g}_a{auto_weight:g}_{reduction}" + (f"_r{ratio:.2f}" if ratio is not None else "")
            score_sweep_rows.append({
                "name": name,
                "student_weight": float(student_weight),
                "auto_weight": float(auto_weight),
                "reduction": reduction,
                "topk_ratio": None if ratio is None else float(ratio),
                "threshold": threshold_variant,
                "precision": metrics_variant["precision"],
                "recall": metrics_variant["recall"],
                "f1": metrics_variant["f1"],
                "auroc": metrics_variant["auroc"],
                "auprc": metrics_variant["auprc"],
                "best_sweep_f1": best_variant["f1"],
            })

score_sweep_df = pd.DataFrame(score_sweep_rows).sort_values(["f1", "auprc", "best_sweep_f1"], ascending=False) if score_sweep_rows else pd.DataFrame()
if not score_sweep_df.empty:
    score_sweep_df.to_csv(evaluation_dir / "score_sweep.csv", index=False)

score_sweep_df.head(10)

score_sweep_path = evaluation_dir / "score_sweep.csv"
if score_sweep_df.empty and score_sweep_path.exists():
    score_sweep_df = pd.read_csv(score_sweep_path)

In [ ]:
if score_sweep_df.empty:
    print("No score sweep results available.")
else:
    default_row = pd.DataFrame([{
        "name": "default_config_score",
        "precision": default_eval_summary["metrics_at_validation_threshold"]["precision"],
        "recall": default_eval_summary["metrics_at_validation_threshold"]["recall"],
        "f1": default_eval_summary["metrics_at_validation_threshold"]["f1"],
        "auroc": default_eval_summary["metrics_at_validation_threshold"]["auroc"],
        "auprc": default_eval_summary["metrics_at_validation_threshold"]["auprc"],
        "best_sweep_f1": default_eval_summary["best_threshold_sweep"]["f1"],
    }])

    plot_df = pd.concat([
        default_row,
        score_sweep_df[["name", "precision", "recall", "f1", "auroc", "auprc", "best_sweep_f1"]]
    ], ignore_index=True).sort_values("f1", ascending=False).head(12)

    display(plot_df)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(plot_df["name"][::-1], plot_df["f1"][::-1])
    ax.set_xlabel("Validation-threshold F1")
    ax.set_title("WideResNet50-2 TS score sweep")
    plt.tight_layout()
    fig.savefig(plots_dir / "score_sweep_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()

## Selected Defect Breakdown

This cell recomputes the defect-type recall for the **best score-sweep row** and saves it beside the evaluation outputs.

In [ ]:
# DEFECT_BREAKDOWN_CELL
score_sweep_path = evaluation_dir / "score_sweep.csv"
if ("score_sweep_df" not in globals()) or score_sweep_df.empty:
    if score_sweep_path.exists():
        score_sweep_df = pd.read_csv(score_sweep_path)
    else:
        raise FileNotFoundError(f"Score sweep file not found: {score_sweep_path}. No retraining needed, but rerun the score-sweep/evaluation cells first.")

selected_variant = score_sweep_df.iloc[0].to_dict()
output_path = evaluation_dir / f"{selected_variant['name']}_defect_breakdown.csv"

if output_path.exists():
    defect_breakdown_df = pd.read_csv(output_path)
else:
    required_globals = ["model", "val_loader", "test_loader", "device", "test_dataset"]
    missing_globals = [name for name in required_globals if name not in globals()]
    if missing_globals:
        raise RuntimeError(
            "No retraining needed. Rerun the notebook setup/evaluation cells so these objects exist: "
            + ", ".join(missing_globals)
        )

    if any(name not in globals() for name in ["val_student_maps", "val_auto_maps", "val_labels", "test_student_maps", "test_auto_maps", "test_labels"]):
        val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(model, val_loader, device, desc="Val maps (defect breakdown)")
        test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(model, test_loader, device, desc="Test maps (defect breakdown)")

    student_weight = float(selected_variant["student_weight"])
    auto_weight = float(selected_variant["auto_weight"])
    reduction = str(selected_variant["reduction"])
    topk_ratio_value = selected_variant.get("topk_ratio", np.nan)
    topk_ratio = None if pd.isna(topk_ratio_value) else float(topk_ratio_value)
    selected_threshold = float(selected_variant["threshold"])

    selected_test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
    selected_test_scores = np.asarray(reduce_map_np(selected_test_map, reduction, topk_ratio)).reshape(-1)

    analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
    analysis_df["score"] = selected_test_scores
    analysis_df["predicted_anomaly"] = (analysis_df["score"] > selected_threshold).astype(int)

    defect_breakdown_df = (
        analysis_df.loc[analysis_df["is_anomaly"] == 1]
        .groupby("defect_type")
        .agg(
            count=("defect_type", "size"),
            detected=("predicted_anomaly", "sum"),
            mean_score=("score", "mean"),
            median_score=("score", "median"),
        )
        .reset_index()
    )
    defect_breakdown_df["detected"] = defect_breakdown_df["detected"].astype(int)
    defect_breakdown_df["missed"] = defect_breakdown_df["count"] - defect_breakdown_df["detected"]
    defect_breakdown_df["recall"] = defect_breakdown_df["detected"] / defect_breakdown_df["count"]
    defect_breakdown_df = defect_breakdown_df.sort_values(["recall", "count", "defect_type"], ascending=[True, False, True]).reset_index(drop=True)
    defect_breakdown_df.to_csv(output_path, index=False)

display(defect_breakdown_df)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(defect_breakdown_df["defect_type"], defect_breakdown_df["recall"], color="#457b9d")
ax.set_ylim(0.0, 1.05)
ax.set_title("Recall by defect type for selected score variant")
ax.set_ylabel("recall")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(plots_dir / "defect_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()